# Convert NetCDF Point Data To GeoParquet

This notebook converts the ALBATROS tide-gauge NetCDF example into GeoParquet. The important steps are to flatten the NetCDF variables into a table and create point geometries from longitude and latitude.

In [1]:
from pathlib import Path


def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root when the kernel starts in a subfolder."""
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / ".git").exists() or (path / "downloaded_data").exists():
            return path
    return start


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "downloaded_data"
DATA_DIR.mkdir(exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Data directory: {DATA_DIR}")

import json

import geopandas as gpd
import pandas as pd
import xarray as xr

NC_FILE = DATA_DIR / "datacollection_ALBATROSS_tidal_amplitude_phase_tide_gauges.nc"
PARQUET_FILE = DATA_DIR / "tidal_amplitude_phase_tide_gauges.parquet"
METADATA_FILE = DATA_DIR / "tidal_amplitude_phase_tide_gauges.parquet.metadata.json"

Repository root: /home/krasen/hackathon-repo
Data directory: /home/krasen/hackathon-repo/downloaded_data


In [2]:
ds = xr.open_dataset(NC_FILE)
ds

<xarray.Dataset> Size: 4kB
Dimensions:    (npoints: 54, ndim: 2)
Dimensions without coordinates: npoints, ndim
Data variables:
    LATITUDE   (npoints) float32 216B ...
    LONGITUDE  (npoints) float32 216B ...
    K1_tide    (npoints, ndim) float32 432B ...
    K2_tide    (npoints, ndim) float32 432B ...
    M2_tide    (npoints, ndim) float32 432B ...
    N2_tide    (npoints, ndim) float32 432B ...
    O1_tide    (npoints, ndim) float32 432B ...
    P1_tide    (npoints, ndim) float32 432B ...
    Q1_tide    (npoints, ndim) float32 432B ...
    S2_tide    (npoints, ndim) float32 432B ...

## Flatten The NetCDF Dataset


In [3]:


df = ds.to_dataframe().reset_index()
longitude_col = next(col for col in ["LONGITUDE", "longitude", "lon"] if col in df.columns)
latitude_col = next(col for col in ["LATITUDE", "latitude", "lat"] if col in df.columns)

df = df.dropna(subset=[longitude_col, latitude_col])
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df[longitude_col], df[latitude_col]),
    crs="EPSG:4326",
)

gdf.head()


,npoints,ndim,LATITUDE,LONGITUDE,K1_tide,K2_tide,M2_tide,N2_tide,O1_tide,P1_tide,Q1_tide,S2_tide,geometry
0,0,0,-70.772003,348.134003,0.260192,0.091117,0.436630,0.066328,0.278369,0.087957,0.066499,0.316667,POINT (348.134 -70.772)
1,0,1,-70.772003,348.134003,358.464691,224.964859,205.193726,189.162949,353.114929,359.157837,343.661377,224.881927,POINT (348.134 -70.772)
2,1,0,-67.570999,291.871002,0.289364,0.054577,0.119572,0.062056,0.233044,0.091737,0.048814,0.189184,POINT (291.871 -67.571)
3,1,1,-67.570999,291.871002,88.067108,71.799179,259.170135,147.328293,75.937294,86.624428,68.358376,65.194763,POINT (291.871 -67.571)
4,2,0,-74.383003,322.350006,0.291561,0.114971,0.567569,0.087266,0.321162,0.097297,0.083776,0.398515,POINT (322.35001 -74.383)


In [4]:
gdf.explore()

In [5]:
# To get effective geo parquet you have to use atleast schema=1.1.0 + sort the data spatially
# Infact if you have more  than 2gbs worth of data its a good idea to partition it spatially, not just sort it
# more info on best practices here: - https://geoparquet.io/
gdf.sort_values('geometry',  inplace=True, ignore_index=True)

gdf.to_parquet(
    PARQUET_FILE,
    engine='pyarrow',
    compression='zstd',
    write_covering_bbox=True, 
    schema_version='1.1.0'
)